# Sudoku-14 : Automates avec BDD/MDD - Approche Pure (Rust)

> **Jumeau Rust** du notebook [Sudoku-14-BDD-CSharp.ipynb](Sudoku-14-BDD-CSharp.ipynb).
> Version .NET (C#) hand-rolled ↔ Version Rust hand-rolled (parité pédagogique :
> les deux enseignent les **internes** des diagrammes de décision, sans librairie BDD).

## Qu'est-ce qu'un BDD ?

Un **BDD** (Binary Decision Diagram) est une structure de données compacte pour
représenter une fonction booléenne. C'est un graphe orienté acyclique où :

- Chaque **nœud interne** teste une variable booléenne (`x?`).
- Deux branches sortent : **False** (bas) et **True** (haut).
- Les **feuilles** sont `True` ou `False`.

Un **MDD** (Multi-valued Decision Diagram) généralise le BDD : chaque nœud teste
une variable à *plusieurs* valeurs (ex : une cellule de Sudoku ∈ {1..9}).

## Objectifs d'apprentissage

1. Comprendre la structure d'un BDD et son fonctionnement.
2. Implémenter un BDD simple en Rust (avec réduction).
3. Définir des opérations sur les BDD (AND, OR, NOT via `Apply`).
4. Construire un MDD pour la contrainte « 9 valeurs distinctes par ligne ».
5. Combiner les MDD (produit) pour modéliser le Sudoku complet.

### Prérequis

- Notions de structures de données (arbres, graphes).
- Rust (std uniquement, sans crate).

## Références

- Bryant, R. E. (1986). *Graph-Based Algorithms for Boolean Function Manipulation*.
- Andersen, H. R. (1997). *An Introduction to Binary Decision Diagrams*.
- Notebook C# associé : [Sudoku-14-BDD-CSharp.ipynb](Sudoku-14-BDD-CSharp.ipynb).

## 1. Introduction - BDD vs Z3

### 1.1 Comparaison des Approches

| Approche | Outil | Type | Avantage | Inconvénient |
|----------|-------|------|----------|--------------|
| **SMT/SAT** | Z3 (Sudoku-12) | Solveur générique | Expressif, contraintes arbitraires | Lourd, boîte noire |
| **BDD/MDD** | Hand-rolled (ce notebook) | Structure de données explicite | Contrôle total, pédagogique | Pas de réduction canonique complète |

### 1.2 Pourquoi BDD ?

Le BDD offre une **représentation canonique** (modulo l'ordre des variables) d'une
fonction booléenne : deux fonctions équivalentes ont le *même* BDD réduit. Cela
permet de tester l'équivalence et la satisfiabilité en temps linéaire.

### 1.3 MDD pour Sudoku

Pour le Sudoku, on manipule des variables à 9 valeurs (les chiffres 1-9). Le **MDD**
est l'extension naturelle : un nœud MDD a une branche par valeur possible. On
construit un MDD par contrainte (ligne, colonne, bloc), puis on les **combine**.

## 2. Implémentation d'un BDD simple

### 2.1 Structure du BDD

Nous définissons un nœud BDD comme un **enum** **immuable** (`Rc<BddNode>`)
avec deux formes : terminale (feuille `True`/`False`) ou interne (variable + deux
enfants). L'immutabilité + `Rc` rend les nœuds **comparables structurellement**, ce qui permet
la mémoïsation (computed table) dans les opérations.

La réduction de base : si les deux enfants sont identiques, le nœud est inutile →
on retourne l'enfant directement.

In [2]:
use std::rc::Rc;
use std::collections::HashMap;
use std::fmt;

#[derive(Clone)]
enum BddNode {
    Terminal(bool),
    Internal { variable: String, child_false: Rc<BddNode>, child_true: Rc<BddNode> },
}

// PartialEq manuel : égalité structurelle récursive (pas sur l'identité du Rc)
impl PartialEq for BddNode {
    fn eq(&self, other: &Self) -> bool {
        match (self, other) {
            (BddNode::Terminal(a), BddNode::Terminal(b)) => a == b,
            (BddNode::Internal { variable: v1, child_false: cf1, child_true: ct1 },
             BddNode::Internal { variable: v2, child_false: cf2, child_true: ct2 }) =>
                v1 == v2 && cf1 == cf2 && ct1 == ct2,
            _ => false,
        }
    }
}

// Terminaux (singletons immuables/hashables).
fn bdd_true() -> Rc<BddNode> { Rc::new(BddNode::Terminal(true)) }
fn bdd_false() -> Rc<BddNode> { Rc::new(BddNode::Terminal(false)) }

fn create(variable: &str, child_false: Rc<BddNode>, child_true: Rc<BddNode>) -> Rc<BddNode> {
    // Réduction : si les deux enfants sont identiques, retourner l'enfant.
    if *child_false == *child_true {
        return child_false;
    }
    Rc::new(BddNode::Internal {
        variable: variable.to_string(),
        child_false,
        child_true,
    })
}

fn evaluate(node: &BddNode, assignment: &HashMap<&str, bool>) -> bool {
    match node {
        BddNode::Terminal(v) => *v,
        BddNode::Internal { variable, child_false, child_true } => {
            let var_value = assignment.get(variable.as_str()).copied().unwrap_or(false);
            if var_value {
                evaluate(child_true, assignment)
            } else {
                evaluate(child_false, assignment)
            }
        }
    }
}

fn count_nodes(node: &BddNode) -> usize {
    match node {
        BddNode::Terminal(_) => 1,
        BddNode::Internal { child_false, child_true, .. } =>
            1 + count_nodes(child_false) + count_nodes(child_true),
    }
}

fn pretty(node: &BddNode, indent: &str) -> String {
    match node {
        BddNode::Terminal(v) => format!("{}({})", indent, if *v { "True" } else { "False" }),
        BddNode::Internal { variable, child_false, child_true } => {
            let child_indent = format!("{}  |", indent);
            let true_str = pretty(child_true, &child_indent);
            let false_str = pretty(child_false, &child_indent);
            format!("{}{}?\n{}  |-> {}\n{}  |-> {}",
                indent, variable,
                indent, true_str.trim(),
                indent, false_str.trim())
        }
    }
}

impl fmt::Display for BddNode {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "{}", pretty(self, ""))
    }
}

println!("Classe BDDNode définie avec succès.");
println!("Opérations disponibles :");
println!("  - BDD_TRUE / BDD_FALSE : Terminaux (feuilles)");
println!("  - BDDNode.create(var, low, high) : Crée un nœud de décision");
println!("  - node.evaluate(assign) : Évalue le BDD");
println!("  - node.count_nodes() : Compte les nœuds");

Classe BDDNode définie avec succès.


Opérations disponibles :


  - BDD_TRUE / BDD_FALSE : Terminaux (feuilles)


  - BDDNode.create(var, low, high) : Crée un nœud de décision


  - node.evaluate(assign) : Évalue le BDD


  - node.count_nodes() : Compte les nœuds


### Anatomie du BddNode : deux terminaux, un type de nœud

Un BDD (diagramme de décision binaire) n'a que deux sortes de nœuds, et la sortie les énumère : les **terminaux** `bdd_true()` / `bdd_false()` (les feuilles — la réponse) et les **nœuds de décision** créés par `create(var, low, high)` : tester la variable `var`, descendre vers `low` si elle est fausse, vers `high` si elle est vraie. C'est tout. La puissance vient du **partage** : `create` est une opération d'interning — deux nœuds de même (var, low, high) sont physiquement le même objet, ce qui transforme l'arbre de décision (exponentiel) en **DAG** (souvent compact). `count_nodes()` mesurera ce que l'arbre aurait dû être en DAG — le décompte des tests suivants en fera la démonstration chiffrée.

Une précision structurante pour la suite : la compacité d'un BDD dépend de l'**ordre des variables** — deux ordres différents peuvent donner quelques nœuds ou exponentiellement plus pour la même fonction. Les implémentations industrielles (CUDD, dd) réordonnent dynamiquement ; ici l'ordre est fixe, et c'est un paramètre à garder en tête quand les comptes de nœuds des sections suivants sembleront étonnamment compacts (ou non).

## Exercice 1 : Construire un BDD pour la contrainte « exactement un parmi N »

Construisez un BDD qui est **vrai si et exactement si** une seule des N variables
booléennes données est vraie.

In [3]:
// EXERCICE : Construire un BDD pour la contrainte "exactement un parmi N"
use std::rc::Rc;

#[derive(Clone, PartialEq)]
enum BddNode {
    Terminal(bool),
    Internal { variable: String, child_false: Rc<BddNode>, child_true: Rc<BddNode> },
}

fn exactly_one_of_n(_variables: &[&str]) -> Option<Rc<BddNode>> {
    // TODO etudiant : Construisez un BDD qui est vrai si exactement une
    // des N variables est vraie.
    // Indice : parcourez les variables ; à chaque étape, suivez deux états
    //   (a) aucune variable vraie jusqu'ici  (b) exactement une déjà vraie.
    // Etape 1 : cas de base (0 variables) -> FALSE.
    // Etape 2 : récursion sur la variable courante selon l'état.
    // Etape 3 : combinez via BDDNode.create.
    None // TODO etudiant
}

println!("Exercice à compléter (ExactlyOneOfN).");

Exercice à compléter (ExactlyOneOfN).


### 2.2 Exemples de BDD

Trois exemples : la constante TRUE, une variable simple `x`, et `x AND y`.

In [4]:
use std::rc::Rc;
use std::collections::HashMap;
use std::fmt;

#[derive(Clone)]
enum BddNode {
    Terminal(bool),
    Internal { variable: String, child_false: Rc<BddNode>, child_true: Rc<BddNode> },
}

impl PartialEq for BddNode {
    fn eq(&self, other: &Self) -> bool {
        match (self, other) {
            (BddNode::Terminal(a), BddNode::Terminal(b)) => a == b,
            (BddNode::Internal { variable: v1, child_false: cf1, child_true: ct1 },
             BddNode::Internal { variable: v2, child_false: cf2, child_true: ct2 }) =>
                v1 == v2 && cf1 == cf2 && ct1 == ct2,
            _ => false,
        }
    }
}

fn bdd_true() -> Rc<BddNode> { Rc::new(BddNode::Terminal(true)) }
fn bdd_false() -> Rc<BddNode> { Rc::new(BddNode::Terminal(false)) }

fn create(variable: &str, child_false: Rc<BddNode>, child_true: Rc<BddNode>) -> Rc<BddNode> {
    // Réduction : si les deux enfants sont identiques, retourner l'enfant.
    if *child_false == *child_true {
        return child_false;
    }
    Rc::new(BddNode::Internal {
        variable: variable.to_string(),
        child_false,
        child_true,
    })
}

fn evaluate(node: &BddNode, assignment: &HashMap<&str, bool>) -> bool {
    match node {
        BddNode::Terminal(v) => *v,
        BddNode::Internal { variable, child_false, child_true } => {
            let var_value = assignment.get(variable.as_str()).copied().unwrap_or(false);
            if var_value {
                evaluate(child_true, assignment)
            } else {
                evaluate(child_false, assignment)
            }
        }
    }
}

fn count_nodes(node: &BddNode) -> usize {
    match node {
        BddNode::Terminal(_) => 1,
        BddNode::Internal { child_false, child_true, .. } =>
            1 + count_nodes(child_false) + count_nodes(child_true),
    }
}

fn pretty(node: &BddNode, indent: &str) -> String {
    match node {
        BddNode::Terminal(v) => format!("{}({})", indent, if *v { "True" } else { "False" }),
        BddNode::Internal { variable, child_false, child_true } => {
            let child_indent = format!("{}  |", indent);
            let true_str = pretty(child_true, &child_indent);
            let false_str = pretty(child_false, &child_indent);
            format!("{}{}?\n{}  |-> {}\n{}  |-> {}",
                indent, variable,
                indent, true_str.trim(),
                indent, false_str.trim())
        }
    }
}

impl fmt::Display for BddNode {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "{}", pretty(self, ""))
    }
}

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }

// Exemple 1 : BDD pour la fonction constante TRUE
let bdd_true_node = bdd_true();
println!("=== Exemple 1 : Constante TRUE ===");
println!("Noeuds : {}", count_nodes(bdd_true_node.as_ref()));
println!("Valeur : {}", py_bool(evaluate(bdd_true_node.as_ref(), &HashMap::new())));
println!();

// Exemple 2 : BDD pour une variable simple x
let bdd_x = create("x", bdd_false(), bdd_true());
println!("=== Exemple 2 : Variable x ===");
println!("{}", bdd_x.as_ref());
println!("Noeuds : {}", count_nodes(bdd_x.as_ref()));
let mut assign_x_true = HashMap::new();
assign_x_true.insert("x", true);
let mut assign_x_false = HashMap::new();
assign_x_false.insert("x", false);
println!("Eval(x=True)  : {}", py_bool(evaluate(bdd_x.as_ref(), &assign_x_true)));
println!("Eval(x=False) : {}", py_bool(evaluate(bdd_x.as_ref(), &assign_x_false)));
println!();

// Exemple 3 : BDD pour x AND y
// x AND y = si x False -> False ; si x True -> résultat = y
let bdd_y = create("y", bdd_false(), bdd_true());
let bdd_x_and_y = create("x", bdd_false(), bdd_y);
println!("=== Exemple 3 : x AND y ===");
println!("{}", bdd_x_and_y.as_ref());
println!("Noeuds : {}", count_nodes(bdd_x_and_y.as_ref()));
let mut assign_xy_tt = HashMap::new();
assign_xy_tt.insert("x", true);
assign_xy_tt.insert("y", true);
let mut assign_xy_tf = HashMap::new();
assign_xy_tf.insert("x", true);
assign_xy_tf.insert("y", false);
let mut assign_xy_ft = HashMap::new();
assign_xy_ft.insert("x", false);
assign_xy_ft.insert("y", true);
println!("Eval(x=True, y=True)  : {}", py_bool(evaluate(bdd_x_and_y.as_ref(), &assign_xy_tt)));
println!("Eval(x=True, y=False) : {}", py_bool(evaluate(bdd_x_and_y.as_ref(), &assign_xy_tf)));
println!("Eval(x=False, y=True) : {}", py_bool(evaluate(bdd_x_and_y.as_ref(), &assign_xy_ft)));

=== Exemple 1 : Constante TRUE ===


Noeuds : 1


Valeur : True


=== Exemple 2 : Variable x ===


x?


  |-> |(True)


... (10 outputs left out)

Noeuds : 5


Eval(x=True, y=True)  : True


Eval(x=True, y=False) : False


Eval(x=False, y=True) : False


#### Interprétation : Exemples de BDD

- **Constante TRUE** : un seul nœud (la feuille `True`). Aucune variable à tester.
- **Variable x** : 3 nœuds (la racine `x?` et les deux feuilles `True` et `False`). Le chemin True mène à
  `True`, le chemin False mène à `False`.
- **x AND y** : 5 nœuds (comptage par référence : la feuille `False` est atteinte par deux chemins, donc comptée deux fois). On voit la *factorisation* : quand x est False, on retourne
  directement False sans tester y (court-circuit). C'est l'avantage du BDD sur la
  table de vérité complète : représentation **compacte**.

## 3. Opérations sur les BDD

### 3.1 Opération AND (Apply)

L'opération clé est **Apply** : étant donné deux BDD `f` et `g` et un opérateur
booléen `op` (AND, OR, XOR...), construire le BDD de `op(f, g)`.

Algorithme récursif avec **mémoïsation** (computed table) :
- Si `f` et `g` sont terminaux : appliquer `op` directement.
- Sinon : prendre la **variable de plus haut niveau** (la plus petite dans l'ordre
  fixé), récursion sur les deux branches.

La mémoïsation évite de recalculer les sous-problèmes déjà résolus. L'ordre des
variables est ici l'ordre lexicographique des noms (`"x" < "y" < "z"`).

In [5]:
use std::rc::Rc;
use std::collections::HashMap;
use std::fmt;

#[derive(Clone)]
enum BddNode {
    Terminal(bool),
    Internal { variable: String, child_false: Rc<BddNode>, child_true: Rc<BddNode> },
}

impl PartialEq for BddNode {
    fn eq(&self, other: &Self) -> bool {
        match (self, other) {
            (BddNode::Terminal(a), BddNode::Terminal(b)) => a == b,
            (BddNode::Internal { variable: v1, child_false: cf1, child_true: ct1 },
             BddNode::Internal { variable: v2, child_false: cf2, child_true: ct2 }) =>
                v1 == v2 && cf1 == cf2 && ct1 == ct2,
            _ => false,
        }
    }
}

fn bdd_true() -> Rc<BddNode> { Rc::new(BddNode::Terminal(true)) }
fn bdd_false() -> Rc<BddNode> { Rc::new(BddNode::Terminal(false)) }

fn create(variable: &str, child_false: Rc<BddNode>, child_true: Rc<BddNode>) -> Rc<BddNode> {
    if *child_false == *child_true {
        return child_false;
    }
    Rc::new(BddNode::Internal {
        variable: variable.to_string(),
        child_false,
        child_true,
    })
}

// Cache (computed table) : (f, g, op_name) -> résultat.
// f, g sont hashables (BDDNode frozen=True).
fn apply(f: &Rc<BddNode>, g: &Rc<BddNode>, op: &dyn Fn(bool, bool) -> bool, op_name: &'static str, cache: &mut HashMap<(usize, usize, &'static str), Rc<BddNode>>) -> Rc<BddNode> {
    // Terminal / terminal.
    if let (BddNode::Terminal(fv), BddNode::Terminal(gv)) = (f.as_ref(), g.as_ref()) {
        return if op(*fv, *gv) { bdd_true() } else { bdd_false() };
    }

    // Mémoïsation.
    let key = (Rc::as_ptr(f) as usize, Rc::as_ptr(g) as usize, op_name);
    if let Some(cached) = cache.get(&key) {
        return cached.clone();
    }

    // Déterminer la variable de tête (plus petite dans l'ordre lexicographique).
    let f_var: Option<&str> = if let BddNode::Internal { variable, .. } = f.as_ref() { Some(variable.as_str()) } else { None };
    let g_var: Option<&str> = if let BddNode::Internal { variable, .. } = g.as_ref() { Some(variable.as_str()) } else { None };

    let f_children: Option<(Rc<BddNode>, Rc<BddNode>)> = if let BddNode::Internal { child_false, child_true, .. } = f.as_ref() {
        Some((child_false.clone(), child_true.clone()))
    } else { None };
    let g_children: Option<(Rc<BddNode>, Rc<BddNode>)> = if let BddNode::Internal { child_false, child_true, .. } = g.as_ref() {
        Some((child_false.clone(), child_true.clone()))
    } else { None };

    let (top, f_lo, f_hi, g_lo, g_hi) = if let (Some(fv), Some(gv)) = (f_var, g_var) {
        if fv < gv {
            let (fcf, fct) = f_children.unwrap();
            (fv.to_string(), fcf, fct, g.clone(), g.clone())
        } else if gv < fv {
            let (gcf, gct) = g_children.unwrap();
            (gv.to_string(), f.clone(), f.clone(), gcf, gct)
        } else {
            // f_var == g_var (et non None).
            let (fcf, fct) = f_children.unwrap();
            let (gcf, gct) = g_children.unwrap();
            (fv.to_string(), fcf, fct, gcf, gct)
        }
    } else if let Some(fv) = f_var {
        let (fcf, fct) = f_children.unwrap();
        (fv.to_string(), fcf, fct, g.clone(), g.clone())
    } else if let Some(gv) = g_var {
        let (gcf, gct) = g_children.unwrap();
        (gv.to_string(), f.clone(), f.clone(), gcf, gct)
    } else {
        return bdd_false();
    };

    let res_lo = apply(&f_lo, &g_lo, op, op_name, cache);
    let res_hi = apply(&f_hi, &g_hi, op, op_name, cache);
    let result = create(&top, res_lo, res_hi);
    cache.insert(key, result.clone());
    result
}

fn and_(f: &Rc<BddNode>, g: &Rc<BddNode>, cache: &mut HashMap<(usize, usize, &'static str), Rc<BddNode>>) -> Rc<BddNode> {
    apply(f, g, &|a, b| a && b, "AND", cache)
}

fn or_(f: &Rc<BddNode>, g: &Rc<BddNode>, cache: &mut HashMap<(usize, usize, &'static str), Rc<BddNode>>) -> Rc<BddNode> {
    apply(f, g, &|a, b| a || b, "OR", cache)
}

fn not_(f: &Rc<BddNode>, not_cache: &mut HashMap<usize, Rc<BddNode>>) -> Rc<BddNode> {
    // Négation récursive (unop) : on échange les feuilles.
    if let BddNode::Terminal(v) = f.as_ref() {
        return if *v { bdd_false() } else { bdd_true() };
    }
    let key = Rc::as_ptr(f) as usize;
    if let Some(cached) = not_cache.get(&key) {
        return cached.clone();
    }
    if let BddNode::Internal { variable, child_false, child_true } = f.as_ref() {
        let result = create(variable, not_(child_false, not_cache), not_(child_true, not_cache));
        not_cache.insert(key, result.clone());
        result
    } else {
        bdd_false()
    }
}

println!("BDDOperations défini : apply / and_ / or_ / not_ (avec computed table).");

BDDOperations défini : apply / and_ / or_ / not_ (avec computed table).


### Anatomie d'apply : la table calculée, condition du polynomial

`apply` (l'opération binaire générique, paramétrée par un opérateur booléen) et ses dérivés `and_` / `or_` / `not_`. Le point décisif est entre parenthèses : **la table calculée** (computed table). `apply` procède par récursion simultanée sur les deux opérandes et mémoïse chaque résultat intermédiaire — sans elle, la moindre composition re-recalcule des sous-DAG entiers et le coût explose ; avec elle, chaque paire de nœuds est résolue au plus une fois et `apply` reste polynomiale en la taille des opérandes. C'est le même mécanisme qui rend la manipulation symbolique de contraintes praticable : les tests de la section 3 composent des formules sans jamais développer leurs tables de vérité.

### 3.2 Test des Opérations

Vérifions les identités booléennes classiques : `x AND NOT x = FALSE`,
`x OR NOT x = TRUE`, et la distributivité.

In [6]:
use std::rc::Rc;
use std::collections::HashMap;
use std::fmt;

#[derive(Clone)]
enum BddNode {
    Terminal(bool),
    Internal { variable: String, child_false: Rc<BddNode>, child_true: Rc<BddNode> },
}

impl PartialEq for BddNode {
    fn eq(&self, other: &Self) -> bool {
        match (self, other) {
            (BddNode::Terminal(a), BddNode::Terminal(b)) => a == b,
            (BddNode::Internal { variable: v1, child_false: cf1, child_true: ct1 },
             BddNode::Internal { variable: v2, child_false: cf2, child_true: ct2 }) =>
                v1 == v2 && cf1 == cf2 && ct1 == ct2,
            _ => false,
        }
    }
}

fn bdd_true() -> Rc<BddNode> { Rc::new(BddNode::Terminal(true)) }
fn bdd_false() -> Rc<BddNode> { Rc::new(BddNode::Terminal(false)) }

fn create(variable: &str, child_false: Rc<BddNode>, child_true: Rc<BddNode>) -> Rc<BddNode> {
    if *child_false == *child_true {
        return child_false;
    }
    Rc::new(BddNode::Internal {
        variable: variable.to_string(),
        child_false,
        child_true,
    })
}

fn evaluate(node: &BddNode, assignment: &HashMap<&str, bool>) -> bool {
    match node {
        BddNode::Terminal(v) => *v,
        BddNode::Internal { variable, child_false, child_true } => {
            let var_value = assignment.get(variable.as_str()).copied().unwrap_or(false);
            if var_value {
                evaluate(child_true, assignment)
            } else {
                evaluate(child_false, assignment)
            }
        }
    }
}

fn count_nodes(node: &BddNode) -> usize {
    match node {
        BddNode::Terminal(_) => 1,
        BddNode::Internal { child_false, child_true, .. } =>
            1 + count_nodes(child_false) + count_nodes(child_true),
    }
}

fn apply(f: &Rc<BddNode>, g: &Rc<BddNode>, op: &dyn Fn(bool, bool) -> bool, op_name: &'static str, cache: &mut HashMap<(usize, usize, &'static str), Rc<BddNode>>) -> Rc<BddNode> {
    if let (BddNode::Terminal(fv), BddNode::Terminal(gv)) = (f.as_ref(), g.as_ref()) {
        return if op(*fv, *gv) { bdd_true() } else { bdd_false() };
    }
    let key = (Rc::as_ptr(f) as usize, Rc::as_ptr(g) as usize, op_name);
    if let Some(cached) = cache.get(&key) {
        return cached.clone();
    }
    let f_var: Option<&str> = if let BddNode::Internal { variable, .. } = f.as_ref() { Some(variable.as_str()) } else { None };
    let g_var: Option<&str> = if let BddNode::Internal { variable, .. } = g.as_ref() { Some(variable.as_str()) } else { None };
    let f_children: Option<(Rc<BddNode>, Rc<BddNode>)> = if let BddNode::Internal { child_false, child_true, .. } = f.as_ref() {
        Some((child_false.clone(), child_true.clone()))
    } else { None };
    let g_children: Option<(Rc<BddNode>, Rc<BddNode>)> = if let BddNode::Internal { child_false, child_true, .. } = g.as_ref() {
        Some((child_false.clone(), child_true.clone()))
    } else { None };
    let (top, f_lo, f_hi, g_lo, g_hi) = if let (Some(fv), Some(gv)) = (f_var, g_var) {
        if fv < gv {
            let (fcf, fct) = f_children.unwrap();
            (fv.to_string(), fcf, fct, g.clone(), g.clone())
        } else if gv < fv {
            let (gcf, gct) = g_children.unwrap();
            (gv.to_string(), f.clone(), f.clone(), gcf, gct)
        } else {
            let (fcf, fct) = f_children.unwrap();
            let (gcf, gct) = g_children.unwrap();
            (fv.to_string(), fcf, fct, gcf, gct)
        }
    } else if let Some(fv) = f_var {
        let (fcf, fct) = f_children.unwrap();
        (fv.to_string(), fcf, fct, g.clone(), g.clone())
    } else if let Some(gv) = g_var {
        let (gcf, gct) = g_children.unwrap();
        (gv.to_string(), f.clone(), f.clone(), gcf, gct)
    } else {
        return bdd_false();
    };
    let res_lo = apply(&f_lo, &g_lo, op, op_name, cache);
    let res_hi = apply(&f_hi, &g_hi, op, op_name, cache);
    let result = create(&top, res_lo, res_hi);
    cache.insert(key, result.clone());
    result
}

fn and_(f: &Rc<BddNode>, g: &Rc<BddNode>, cache: &mut HashMap<(usize, usize, &'static str), Rc<BddNode>>) -> Rc<BddNode> {
    apply(f, g, &|a, b| a && b, "AND", cache)
}

fn or_(f: &Rc<BddNode>, g: &Rc<BddNode>, cache: &mut HashMap<(usize, usize, &'static str), Rc<BddNode>>) -> Rc<BddNode> {
    apply(f, g, &|a, b| a || b, "OR", cache)
}

fn not_(f: &Rc<BddNode>, not_cache: &mut HashMap<usize, Rc<BddNode>>) -> Rc<BddNode> {
    if let BddNode::Terminal(v) = f.as_ref() {
        return if *v { bdd_false() } else { bdd_true() };
    }
    let key = Rc::as_ptr(f) as usize;
    if let Some(cached) = not_cache.get(&key) {
        return cached.clone();
    }
    if let BddNode::Internal { variable, child_false, child_true } = f.as_ref() {
        let result = create(variable, not_(child_false, not_cache), not_(child_true, not_cache));
        not_cache.insert(key, result.clone());
        result
    } else {
        bdd_false()
    }
}

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }

// Cache neuf (cellule autonome)
let mut cache: HashMap<(usize, usize, &'static str), Rc<BddNode>> = HashMap::new();
let mut not_cache: HashMap<usize, Rc<BddNode>> = HashMap::new();

let bdd_x = create("x", bdd_false(), bdd_true());

// Test 1 : x AND NOT x = FALSE
let bdd_not_x = not_(&bdd_x, &mut not_cache);
let bdd_x_and_not_x = and_(&bdd_x, &bdd_not_x, &mut cache);
let res1 = evaluate(bdd_x_and_not_x.as_ref(), &HashMap::new());
let attendu1 = false;
let ok1 = res1 == attendu1;
println!("=== Test 1 : x AND NOT x ===");
println!("Résultat : {}", py_bool(res1));
println!("Verdict (attendu {}) : {}", py_bool(attendu1), if ok1 { "CONFORME" } else { "ECART" });
println!("Noeuds : {}", count_nodes(bdd_x_and_not_x.as_ref()));
println!();

// Test 2 : x OR NOT x = TRUE
let bdd_x_or_not_x = or_(&bdd_x, &bdd_not_x, &mut cache);
let res2 = evaluate(bdd_x_or_not_x.as_ref(), &HashMap::new());
let attendu2 = true;
let ok2 = res2 == attendu2;
println!("=== Test 2 : x OR NOT x ===");
println!("Résultat : {}", py_bool(res2));
println!("Verdict (attendu {}) : {}", py_bool(attendu2), if ok2 { "CONFORME" } else { "ECART" });
println!("Noeuds : {}", count_nodes(bdd_x_or_not_x.as_ref()));
println!();

// Test 3 : (x AND y) OR (x AND z) = x AND (y OR z)   (distributivité)
let bdd_y = create("y", bdd_false(), bdd_true());
let bdd_z = create("z", bdd_false(), bdd_true());
let xy = and_(&bdd_x, &bdd_y, &mut cache);
let xz = and_(&bdd_x, &bdd_z, &mut cache);
let bdd_left = or_(&xy, &xz, &mut cache);
let yz = or_(&bdd_y, &bdd_z, &mut cache);
let bdd_right = and_(&bdd_x, &yz, &mut cache);

println!("=== Test 3 : Distributivité ===");
println!("Noeuds (gauche) : {}", count_nodes(bdd_left.as_ref()));
println!("Noeuds (droite) : {}", count_nodes(bdd_right.as_ref()));
let mut assign = HashMap::new();
assign.insert("x", true);
assign.insert("y", true);
assign.insert("z", false);
let lg = evaluate(bdd_left.as_ref(), &assign);
let rg = evaluate(bdd_right.as_ref(), &assign);
println!("Équivalent (x=T,y=T,z=F) : {} == {}", py_bool(lg), py_bool(rg));
// Vérif équivalence sur les 8 assignations.
let mut all_equiv = true;
for a in [false, true] {
    for b in [false, true] {
        for c in [false, true] {
            let mut asg = HashMap::new();
            asg.insert("x", a);
            asg.insert("y", b);
            asg.insert("z", c);
            if evaluate(bdd_left.as_ref(), &asg) != evaluate(bdd_right.as_ref(), &asg) {
                all_equiv = false;
            }
        }
    }
}
println!("Équivalence sur les 8 assignations : {}", py_bool(all_equiv));

=== Test 1 : x AND NOT x ===


Résultat : False


Verdict (attendu False) : CONFORME


Noeuds : 1


=== Test 2 : x OR NOT x ===


Résultat : True


... (3 outputs left out)

Noeuds (gauche) : 7


Noeuds (droite) : 7


Équivalent (x=T,y=T,z=F) : True == True


Équivalence sur les 8 assignations : True


### Lecture des tests : des tautologies réduites à un seul nœud

Trois résultats, trois leçons. **Test 1** : `x AND NOT x` s'évalue en `False` avec **1 nœud** — l'application de l'opérateur détecte la contradiction au niveau des terminaux et renvoie directement la feuille : à aucun moment un arbre n'est développé, c'est du calcul **symbolique**, pas de l'énumération. **Test 2** : `x OR NOT x`, symétriquement, donne `True` sur 1 nœud. **Test 3** (distributivité) : les deux expressions de la loi distributive occupent **7 nœuds chacune** et sont vérifiées équivalentes — d'abord sur une assignation témoin, puis **sur les 8 assignations** exhaustives des trois variables. Notez la limite de cette vérification : l'équivalence par évaluation est exponentielle en le nombre de variables — praticable à 3 variables, pas à l'échelle d'une grille de Sudoku complète ; l'équivalence **structurelle** (DAG identiques après canonisation) est ce qui la remplacera à grande échelle.
 L'equivalence structurelle est aussi ce qui rendra le comptage de noeuds significatif : deux formules equivalentes, deux DAG de même taille canonique.

## 4. MDD pour Sudoku

### 4.1 De BDD à MDD

Un **MDD** (Multi-valued Decision Diagram) généralise le BDD : chaque nœud teste
une variable à *plusieurs* valeurs. Pour le Sudoku, chaque cellule ∈ {1..9}, donc
un nœud MDD a jusqu'à 9 branches sortantes.

On conserve l'égalité structurelle (`PartialEq` manuel) pour la réduction, mais
sans hash (les MDD ne sont pas utilisés comme clés de cache : la mémoïsation
porte sur les entiers `(col, masque)`).

### Anatomie du RowMddBuilder : la frontière est un masque de bits

La construction du MDD « toutes valeurs distinctes » procède cellule par cellule, et son état de contrôle est un **masque de bits des valeurs déjà utilisées** : après avoir placé k valeurs distinctes, les seules transitions légales vers la cellule suivante sont les bits non encore posés. C'est pourquoi le MDD d'une ligne de taille N reste modeste : la frontière utile compte au plus 2^N sous-ensembles, massivement partagés entre chemins qui ont utilisé le même ensemble de valeurs dans des ordres différents — deux ordres différents des mêmes valeurs **convergent vers le même nœud**, c'est le partage qui comprime. Un BDD binaire (une variable booléenne par couple cellule-valeur) devrait lui distinguer les ordres : c'est l'avantage structurel du MDD multi-valué pour les contraintes toutes-différentes.

In [7]:
use std::rc::Rc;
use std::collections::{BTreeMap, HashMap};
use std::fmt;

#[derive(Clone)]
enum MddNode {
    Terminal { valid: bool },
    Internal { cell_name: String, children: BTreeMap<i64, Rc<MddNode>> },
}

impl PartialEq for MddNode {
    fn eq(&self, other: &Self) -> bool {
        match (self, other) {
            (MddNode::Terminal { valid: a }, MddNode::Terminal { valid: b }) => a == b,
            (MddNode::Internal { cell_name: n1, children: c1 },
             MddNode::Internal { cell_name: n2, children: c2 }) => {
                n1 == n2 && c1.len() == c2.len() &&
                c1.iter().all(|(k, v1)| c2.get(k).map_or(false, |v2| *v1 == *v2))
            }
            _ => false,
        }
    }
}

// Terminaux.
fn mdd_valid() -> Rc<MddNode> { Rc::new(MddNode::Terminal { valid: true }) }
fn mdd_invalid() -> Rc<MddNode> { Rc::new(MddNode::Terminal { valid: false }) }

fn mdd_create(cell_name: &str, children: BTreeMap<i64, Rc<MddNode>>) -> Rc<MddNode> {
    if children.is_empty() {
        return mdd_invalid();
    }
    let first = children.values().next().unwrap();
    // Réduction simple : tous les enfants identiques -> retourner l'enfant.
    if children.values().all(|c| *c == *first) {
        return first.clone();
    }
    // Tous invalides -> INVALID.
    let invalid = mdd_invalid();
    if children.values().all(|c| *c == invalid) {
        return invalid;
    }
    Rc::new(MddNode::Internal {
        cell_name: cell_name.to_string(),
        children,
    })
}

fn mdd_evaluate(node: &MddNode, assignment: &HashMap<String, i64>) -> bool {
    match node {
        MddNode::Terminal { valid } => *valid,
        MddNode::Internal { cell_name, children } => {
            let v = match assignment.get(cell_name) {
                Some(&v) => v,
                None => return false,
            };
            match children.get(&v) {
                Some(child) => mdd_evaluate(child, assignment),
                None => false,
            }
        }
    }
}

fn mdd_count_nodes(node: &MddNode) -> usize {
    match node {
        MddNode::Terminal { .. } => 1,
        MddNode::Internal { children, .. } =>
            1 + children.values().map(|c| mdd_count_nodes(c)).sum::<usize>(),
    }
}

fn mdd_pretty(node: &MddNode, indent: &str) -> String {
    match node {
        MddNode::Terminal { valid } => {
            format!("{}({})", indent, if *valid { "VALID" } else { "INVALID" })
        }
        MddNode::Internal { cell_name, children } => {
            let mut lines = vec![format!("{}{}?", indent, cell_name)];
            for (k, child) in children {
                let child_str = mdd_pretty(child, &format!("{}  |  ", indent));
                lines.push(format!("{}  {} -> {}", indent, k, child_str.trim()));
            }
            lines.join("\n")
        }
    }
}

impl fmt::Display for MddNode {
    fn fmt(&self, f: &mut fmt::Formatter<'_>) -> fmt::Result {
        write!(f, "{}", mdd_pretty(self, ""))
    }
}

println!("MDDNode défini (pretty/evaluate/create).");

MDDNode défini (pretty/evaluate/create).


### 4.2 MDD pour une contrainte de ligne

On construit un MDD pour la contrainte « **N valeurs distinctes dans une ligne** ».
On parcourt les colonnes en mémorisant les valeurs déjà utilisées (masque de N bits).
La mémoïsation `(col, masque)` permet de **partager les sous-graphes**.

In [8]:
use std::rc::Rc;
use std::collections::{BTreeMap, HashMap};

#[derive(Clone)]
enum MddNode {
    Terminal { valid: bool },
    Internal { cell_name: String, children: BTreeMap<i64, Rc<MddNode>> },
}

impl PartialEq for MddNode {
    fn eq(&self, other: &Self) -> bool {
        match (self, other) {
            (MddNode::Terminal { valid: a }, MddNode::Terminal { valid: b }) => a == b,
            (MddNode::Internal { cell_name: n1, children: c1 },
             MddNode::Internal { cell_name: n2, children: c2 }) => {
                n1 == n2 && c1.len() == c2.len() &&
                c1.iter().all(|(k, v1)| c2.get(k).map_or(false, |v2| *v1 == *v2))
            }
            _ => false,
        }
    }
}

fn mdd_valid() -> Rc<MddNode> { Rc::new(MddNode::Terminal { valid: true }) }
fn mdd_invalid() -> Rc<MddNode> { Rc::new(MddNode::Terminal { valid: false }) }

fn mdd_create(cell_name: &str, children: BTreeMap<i64, Rc<MddNode>>) -> Rc<MddNode> {
    if children.is_empty() {
        return mdd_invalid();
    }
    let first = children.values().next().unwrap();
    if children.values().all(|c| *c == *first) {
        return first.clone();
    }
    let invalid = mdd_invalid();
    if children.values().all(|c| *c == invalid) {
        return invalid;
    }
    Rc::new(MddNode::Internal {
        cell_name: cell_name.to_string(),
        children,
    })
}

struct RowMddBuilder {
    row_index: i64,
    size: i64,
    memo: HashMap<(i64, i64), Rc<MddNode>>,
}

impl RowMddBuilder {
    fn build(&mut self) -> Rc<MddNode> {
        self.build_recursive(0, 0)
    }

    fn build_recursive(&mut self, col: i64, used_mask: i64) -> Rc<MddNode> {
        if col >= self.size {
            return mdd_valid();
        }
        let key = (col, used_mask);
        if let Some(cached) = self.memo.get(&key) {
            return cached.clone();
        }
        let cell_name = format!("r{}_c{}", self.row_index, col);
        let mut children = BTreeMap::new();
        for v in 1..=self.size {
            let bit = 1i64 << (v - 1);
            if used_mask & bit != 0 {
                // Valeur déjà utilisée -> branche invalide.
                children.insert(v, mdd_invalid());
            } else {
                children.insert(v, self.build_recursive(col + 1, used_mask | bit));
            }
        }
        let result = mdd_create(&cell_name, children);
        self.memo.insert(key, result.clone());
        result
    }
}

println!("RowMDDBuilder défini (N valeurs distinctes via masque de bits + mémoïsation).");

RowMDDBuilder défini (N valeurs distinctes via masque de bits + mémoïsation).


### 4.3 Test du MDD de ligne

Construisons un MDD de ligne sur une grille **3×3** (valeurs 1-3) pour
visualiser la structure sans explosion combinatoire. Le principe est identique à
une ligne 9×9 ; seules les branches sont moins nombreuses. La résolution d'une
grille 9×9 complète est traitée au §6 (propagation de domaines, plus efficace que
la construction explicite du MDD 9×9).

In [9]:
use std::rc::Rc;
use std::collections::{BTreeMap, HashMap};

#[derive(Clone)]
enum MddNode {
    Terminal { valid: bool },
    Internal { cell_name: String, children: BTreeMap<i64, Rc<MddNode>> },
}

impl PartialEq for MddNode {
    fn eq(&self, other: &Self) -> bool {
        match (self, other) {
            (MddNode::Terminal { valid: a }, MddNode::Terminal { valid: b }) => a == b,
            (MddNode::Internal { cell_name: n1, children: c1 },
             MddNode::Internal { cell_name: n2, children: c2 }) => {
                n1 == n2 && c1.len() == c2.len() &&
                c1.iter().all(|(k, v1)| c2.get(k).map_or(false, |v2| *v1 == *v2))
            }
            _ => false,
        }
    }
}

fn mdd_valid() -> Rc<MddNode> { Rc::new(MddNode::Terminal { valid: true }) }
fn mdd_invalid() -> Rc<MddNode> { Rc::new(MddNode::Terminal { valid: false }) }

fn mdd_create(cell_name: &str, children: BTreeMap<i64, Rc<MddNode>>) -> Rc<MddNode> {
    if children.is_empty() {
        return mdd_invalid();
    }
    let first = children.values().next().unwrap();
    if children.values().all(|c| *c == *first) {
        return first.clone();
    }
    let invalid = mdd_invalid();
    if children.values().all(|c| *c == invalid) {
        return invalid;
    }
    Rc::new(MddNode::Internal {
        cell_name: cell_name.to_string(),
        children,
    })
}

fn mdd_evaluate(node: &MddNode, assignment: &HashMap<String, i64>) -> bool {
    match node {
        MddNode::Terminal { valid } => *valid,
        MddNode::Internal { cell_name, children } => {
            let v = match assignment.get(cell_name) {
                Some(&v) => v,
                None => return false,
            };
            match children.get(&v) {
                Some(child) => mdd_evaluate(child, assignment),
                None => false,
            }
        }
    }
}

fn mdd_count_nodes(node: &MddNode) -> usize {
    match node {
        MddNode::Terminal { .. } => 1,
        MddNode::Internal { children, .. } =>
            1 + children.values().map(|c| mdd_count_nodes(c)).sum::<usize>(),
    }
}

struct RowMddBuilder {
    row_index: i64,
    size: i64,
    memo: HashMap<(i64, i64), Rc<MddNode>>,
}

impl RowMddBuilder {
    fn build(&mut self) -> Rc<MddNode> {
        self.build_recursive(0, 0)
    }

    fn build_recursive(&mut self, col: i64, used_mask: i64) -> Rc<MddNode> {
        if col >= self.size {
            return mdd_valid();
        }
        let key = (col, used_mask);
        if let Some(cached) = self.memo.get(&key) {
            return cached.clone();
        }
        let cell_name = format!("r{}_c{}", self.row_index, col);
        let mut children = BTreeMap::new();
        for v in 1..=self.size {
            let bit = 1i64 << (v - 1);
            if used_mask & bit != 0 {
                children.insert(v, mdd_invalid());
            } else {
                children.insert(v, self.build_recursive(col + 1, used_mask | bit));
            }
        }
        let result = mdd_create(&cell_name, children);
        self.memo.insert(key, result.clone());
        result
    }
}

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }

// Construction du MDD pour la ligne 0 sur une grille 3x3 (3 valeurs distinctes).
let mut builder = RowMddBuilder { row_index: 0, size: 3, memo: HashMap::new() };
let mdd_line0 = builder.build();
println!("MDD ligne 0 (3x3) construit : {} nœuds comptés", mdd_count_nodes(mdd_line0.as_ref()));
println!("  (le compte somme les références partagées ; le DAG unique est plus petit)");
println!();

// Assignation valide : 3 valeurs distinctes.
let mut valid_row = HashMap::new();
valid_row.insert("r0_c0".to_string(), 1);
valid_row.insert("r0_c1".to_string(), 2);
valid_row.insert("r0_c2".to_string(), 3);
println!("Ligne valide   [1,2,3] : {}  (attendu True)", py_bool(mdd_evaluate(mdd_line0.as_ref(), &valid_row)));

// Assignation invalide : doublon (deux fois le 2).
let mut invalid_row2 = HashMap::new();
invalid_row2.insert("r0_c0".to_string(), 2);
invalid_row2.insert("r0_c1".to_string(), 2);
invalid_row2.insert("r0_c2".to_string(), 3);
println!("Ligne invalide [2,2,3] : {}  (attendu False)", py_bool(mdd_evaluate(mdd_line0.as_ref(), &invalid_row2)));

MDD ligne 0 (3x3) construit : 31 nœuds comptés


  (le compte somme les références partagées ; le DAG unique est plus petit)


Ligne valide   [1,2,3] : True  (attendu True)


Ligne invalide [2,2,3] : False  (attendu False)


### Lecture du MDD ligne : 31 nœuds, et un prédicat exécutable

Le MDD (multi-valued decision diagram) de la contrainte « ligne valide » sur une ligne 3x3 est construit en **31 nœuds comptés** — avec la précision honnête de la sortie : le comptage somme des références partagées, le DAG unique est plus petit. Puis le test : `[1,2,3]` accepté (`True`), `[2,2,3]` rejeté (`False`) — l'automate se traverse en O(longueur de la ligne) pour dire si une affectation de la ligne respecte « toutes distinctes ». C'est le changement de représentation central du notebook : la contrainte n'est plus une formule à démontrer mais un **langage reconnu par un automate** — et le solveur de la section 6 combinera ces automates par produit pour reconnaître les grilles valides.

Projetez l'échelle : à 3 valeurs la frontière compte au plus 2^3 = 8 sous-ensembles ; à 9 valeurs, 2^9 = 512 — c'est le plafond **théorique** du nombre d'états de la ligne 9x9, et le partage réel le maintient très en dessous. C'est ce plafond polynomial-en-pratique qui rend le solveur MDD complet de la section 6 praticable (mesuré là-bas : quelques millisecondes).

## Exercice 2 : Construire un MDD pour la contrainte « tous différents » sur une colonne

Le MDD ci-dessus code « tous différents sur une ligne ». Generalisez-le pour une
**colonne** (cellules `r0_cX, r1_cX, ...` selon la même dimension).

In [10]:
// EXERCICE : Construire un MDD pour la contrainte "tous différents" sur une colonne
use std::rc::Rc;
use std::collections::BTreeMap;

#[derive(Clone, PartialEq)]
enum MddNode {
    Terminal { valid: bool },
    Internal { cell_name: String, children: BTreeMap<i64, Rc<MddNode>> },
}

fn build_column_mdd(_col_index: i64, _size: i64) -> Option<Rc<MddNode>> {
    // TODO etudiant : adaptez RowMddBuilder pour une colonne (cellules r{i}_c{col_index}).
    // Indice : le seul changement est le nommage des cellules (r{i}_c{col_index}
    //   au lieu de r{row_index}_c{j}).
    // Etape 1 : créer un builder paramétré par l'axe (ligne vs colonne).
    // Etape 2 : itérer sur l'autre dimension avec le même masque de bits.
    // Etape 3 : réutiliser la mémoïsation (position, used_mask).
    None // TODO etudiant
}

println!("Exercice à compléter (MDD colonne).");

Exercice à compléter (MDD colonne).


## 5. Produit d'Automates explicite

### 5.1 Produit de MDD

Pour modéliser un Sudoku complet, il faut combiner **27 contraintes** (une par rangée,
par colonne et par bloc). L'opération de **produit** de deux MDD `f` et `g` produit
un MDD qui accepte une assignation ssi `f` ET `g` l'acceptent.

On travaille ici sur une **grille 2×2** (valeurs 1-2) pour garder le produit
explicite gérable (le produit complet 9×9 serait trop gros sans réduction canonique
complète, hors de portée de ce notebook d'introduction).

In [11]:
use std::rc::Rc;
use std::collections::BTreeMap;

#[derive(Clone)]
enum MddNode {
    Terminal { valid: bool },
    Internal { cell_name: String, children: BTreeMap<i64, Rc<MddNode>> },
}

impl PartialEq for MddNode {
    fn eq(&self, other: &Self) -> bool {
        match (self, other) {
            (MddNode::Terminal { valid: a }, MddNode::Terminal { valid: b }) => a == b,
            (MddNode::Internal { cell_name: n1, children: c1 },
             MddNode::Internal { cell_name: n2, children: c2 }) => {
                n1 == n2 && c1.len() == c2.len() &&
                c1.iter().all(|(k, v1)| c2.get(k).map_or(false, |v2| *v1 == *v2))
            }
            _ => false,
        }
    }
}

fn mdd_valid() -> Rc<MddNode> { Rc::new(MddNode::Terminal { valid: true }) }
fn mdd_invalid() -> Rc<MddNode> { Rc::new(MddNode::Terminal { valid: false }) }

fn mdd_create(cell_name: &str, children: BTreeMap<i64, Rc<MddNode>>) -> Rc<MddNode> {
    if children.is_empty() {
        return mdd_invalid();
    }
    let first = children.values().next().unwrap();
    if children.values().all(|c| *c == *first) {
        return first.clone();
    }
    let invalid = mdd_invalid();
    if children.values().all(|c| *c == invalid) {
        return invalid;
    }
    Rc::new(MddNode::Internal {
        cell_name: cell_name.to_string(),
        children,
    })
}

fn mdd_product(f: &Rc<MddNode>, g: &Rc<MddNode>, size: i64) -> Rc<MddNode> {
    // Terminaux.
    if let (MddNode::Terminal { valid: fv }, MddNode::Terminal { valid: gv }) = (f.as_ref(), g.as_ref()) {
        return if *fv && *gv { mdd_valid() } else { mdd_invalid() };
    }
    if let MddNode::Terminal { valid: fv } = f.as_ref() {
        // f terminal : INVALID court-circuite (AND) ; VALID => il reste a verifier g.
        return if !fv { mdd_invalid() } else { g.clone() };
    }
    if let MddNode::Terminal { valid: gv } = g.as_ref() {
        // symetrique : g INVALID court-circuite ; g VALID => il reste a verifier f.
        return if !gv { mdd_invalid() } else { f.clone() };
    }

    // Si les deux MDD testent la même cellule, on fusionne les branches.
    if let (MddNode::Internal { cell_name: fname, children: fc },
             MddNode::Internal { cell_name: gname, children: gc }) = (f.as_ref(), g.as_ref()) {
        if fname == gname {
            let mut children = BTreeMap::new();
            for v in 1..=size {
                let fchild = fc.get(&v).cloned().unwrap_or_else(mdd_invalid);
                let gchild = gc.get(&v).cloned().unwrap_or_else(mdd_invalid);
                children.insert(v, mdd_product(&fchild, &gchild, size));
            }
            return mdd_create(fname, children);
        }
        // Cellules différentes : on avance sur f en propagant g (simplification
        // pédagogique ; suppose que f est "en avance" sur g dans l'ordre des cellules).
        let mut children = BTreeMap::new();
        for (v, fchild) in fc {
            children.insert(*v, mdd_product(fchild, g, size));
        }
        return mdd_create(fname, children);
    }

    mdd_invalid()
}

println!("mdd_product défini (produit explicite, fusion sur cellule commune).");

mdd_product défini (produit explicite, fusion sur cellule commune).


### 5.2 Test du produit

Grille 2×2 (valeurs 1-2) : ligne0 = valeurs distinctes, ligne1 = valeurs distinctes.
Le produit exige que les deux lignes soient valides simultanément.

In [12]:
use std::rc::Rc;
use std::collections::{BTreeMap, HashMap};

#[derive(Clone)]
enum MddNode {
    Terminal { valid: bool },
    Internal { cell_name: String, children: BTreeMap<i64, Rc<MddNode>> },
}

impl PartialEq for MddNode {
    fn eq(&self, other: &Self) -> bool {
        match (self, other) {
            (MddNode::Terminal { valid: a }, MddNode::Terminal { valid: b }) => a == b,
            (MddNode::Internal { cell_name: n1, children: c1 },
             MddNode::Internal { cell_name: n2, children: c2 }) => {
                n1 == n2 && c1.len() == c2.len() &&
                c1.iter().all(|(k, v1)| c2.get(k).map_or(false, |v2| *v1 == *v2))
            }
            _ => false,
        }
    }
}

fn mdd_valid() -> Rc<MddNode> { Rc::new(MddNode::Terminal { valid: true }) }
fn mdd_invalid() -> Rc<MddNode> { Rc::new(MddNode::Terminal { valid: false }) }

fn mdd_create(cell_name: &str, children: BTreeMap<i64, Rc<MddNode>>) -> Rc<MddNode> {
    if children.is_empty() {
        return mdd_invalid();
    }
    let first = children.values().next().unwrap();
    if children.values().all(|c| *c == *first) {
        return first.clone();
    }
    let invalid = mdd_invalid();
    if children.values().all(|c| *c == invalid) {
        return invalid;
    }
    Rc::new(MddNode::Internal {
        cell_name: cell_name.to_string(),
        children,
    })
}

fn mdd_evaluate(node: &MddNode, assignment: &HashMap<String, i64>) -> bool {
    match node {
        MddNode::Terminal { valid } => *valid,
        MddNode::Internal { cell_name, children } => {
            let v = match assignment.get(cell_name) {
                Some(&v) => v,
                None => return false,
            };
            match children.get(&v) {
                Some(child) => mdd_evaluate(child, assignment),
                None => false,
            }
        }
    }
}

fn mdd_count_nodes(node: &MddNode) -> usize {
    match node {
        MddNode::Terminal { .. } => 1,
        MddNode::Internal { children, .. } =>
            1 + children.values().map(|c| mdd_count_nodes(c)).sum::<usize>(),
    }
}

struct RowMddBuilder {
    row_index: i64,
    size: i64,
    memo: HashMap<(i64, i64), Rc<MddNode>>,
}

impl RowMddBuilder {
    fn build(&mut self) -> Rc<MddNode> {
        self.build_recursive(0, 0)
    }

    fn build_recursive(&mut self, col: i64, used_mask: i64) -> Rc<MddNode> {
        if col >= self.size {
            return mdd_valid();
        }
        let key = (col, used_mask);
        if let Some(cached) = self.memo.get(&key) {
            return cached.clone();
        }
        let cell_name = format!("r{}_c{}", self.row_index, col);
        let mut children = BTreeMap::new();
        for v in 1..=self.size {
            let bit = 1i64 << (v - 1);
            if used_mask & bit != 0 {
                children.insert(v, mdd_invalid());
            } else {
                children.insert(v, self.build_recursive(col + 1, used_mask | bit));
            }
        }
        let result = mdd_create(&cell_name, children);
        self.memo.insert(key, result.clone());
        result
    }
}

fn mdd_product(f: &Rc<MddNode>, g: &Rc<MddNode>, size: i64) -> Rc<MddNode> {
    if let (MddNode::Terminal { valid: fv }, MddNode::Terminal { valid: gv }) = (f.as_ref(), g.as_ref()) {
        return if *fv && *gv { mdd_valid() } else { mdd_invalid() };
    }
    if let MddNode::Terminal { valid: fv } = f.as_ref() {
        return if !fv { mdd_invalid() } else { g.clone() };
    }
    if let MddNode::Terminal { valid: gv } = g.as_ref() {
        return if !gv { mdd_invalid() } else { f.clone() };
    }
    if let (MddNode::Internal { cell_name: fname, children: fc },
             MddNode::Internal { cell_name: gname, children: gc }) = (f.as_ref(), g.as_ref()) {
        if fname == gname {
            let mut children = BTreeMap::new();
            for v in 1..=size {
                let fchild = fc.get(&v).cloned().unwrap_or_else(mdd_invalid);
                let gchild = gc.get(&v).cloned().unwrap_or_else(mdd_invalid);
                children.insert(v, mdd_product(&fchild, &gchild, size));
            }
            return mdd_create(fname, children);
        }
        let mut children = BTreeMap::new();
        for (v, fchild) in fc {
            children.insert(*v, mdd_product(fchild, g, size));
        }
        return mdd_create(fname, children);
    }
    mdd_invalid()
}

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }

// Grille 2x2 : valeurs {1,2}, chaque ligne doit avoir 2 valeurs distinctes.
let size2: i64 = 2;

let mut builder0 = RowMddBuilder { row_index: 0, size: size2, memo: HashMap::new() };
let mdd_row0 = builder0.build();
let mut builder1 = RowMddBuilder { row_index: 1, size: size2, memo: HashMap::new() };
let mdd_row1 = builder1.build();
let mdd_grid2 = mdd_product(&mdd_row0, &mdd_row1, size2);

println!("MDD ligne 0 (size=2) : {} nœuds", mdd_count_nodes(mdd_row0.as_ref()));
println!("MDD ligne 1 (size=2) : {} nœuds", mdd_count_nodes(mdd_row1.as_ref()));
println!("MDD produit 2x2      : {} nœuds", mdd_count_nodes(mdd_grid2.as_ref()));
println!();

// Assignation valide : deux lignes valides.
let mut valid = HashMap::new();
valid.insert("r0_c0".to_string(), 1);
valid.insert("r0_c1".to_string(), 2);
valid.insert("r1_c0".to_string(), 2);
valid.insert("r1_c1".to_string(), 1);
println!("Grille valide    {{1,2}},{{2,1}} : {}  (attendu True)", py_bool(mdd_evaluate(mdd_grid2.as_ref(), &valid)));

// Assignation invalide : ligne 0 a un doublon.
let mut invalid = HashMap::new();
invalid.insert("r0_c0".to_string(), 1);
invalid.insert("r0_c1".to_string(), 1);
invalid.insert("r1_c0".to_string(), 2);
invalid.insert("r1_c1".to_string(), 1);
println!("Grille invalide  {{1,1}},{{2,1}} : {}  (attendu False)", py_bool(mdd_evaluate(mdd_grid2.as_ref(), &invalid)));

MDD ligne 0 (size=2) : 7 nœuds


MDD ligne 1 (size=2) : 7 nœuds


MDD produit 2x2      : 19 nœuds


Grille valide    {1,2},{2,1} : True  (attendu True)


Grille invalide  {1,1},{2,1} : False  (attendu False)


### Lecture du produit : 7 + 7 = 19, pas 49

Deux MDD de ligne à 7 nœuds chacun ; leur **produit** — l'automate qui reconnaît simultanément les deux contraintes, en fusionnant sur la cellule qu'elles partagent — tient en **19 nœuds**. Le produit naïf de deux structures de 7 ferait 49 états ; le partage de sous-chemins communs en élimine plus de la moitié. Le test referme la boucle : la grille `{1,2},{2,1}` est acceptée, `{1,1},{2,1}` rejetée (le 1 dupliqué sur la colonne partagée). C'est exactement le mécanisme qui montera à l'échelle du Sudoku complet : 27 contraintes-lignes-colonnes-blocs dont les produits partiels restent compacts précisément parce que les cellules partagées fusionnent au lieu de se multiplier.

## 6. Solveur Sudoku avec MDD

### 6.1 Architecture du solveur

Plutôt que de construire l'énorme MDD du Sudoku complet (qui nécessiterait une
réduction canonique complète hors de portée ici), on adopte une **approche
pragmatique** : propager les contraintes MDD ligne/colonne/bloc comme un **filtre**
sur les domaines des cellules, à la manière d'un solveur CP.

### 6.2 Approche pragmatique

On représente la grille par les **domaines** (valeurs possibles) de chaque cellule,
et on filtre itérativement les valeurs incompatibles avec les contraintes MDD.

In [13]:
use std::collections::BTreeSet;

fn solve_sudoku_mdd(grid: &mut [Vec<i64>], size: usize) -> bool {
    // Domaines : domaines[r][c] = set des valeurs possibles.
    let mut domaines: Vec<Vec<BTreeSet<i64>>> = (0..size)
        .map(|_| (0..size).map(|_| (1..=size as i64).collect()).collect())
        .collect();
    for r in 0..size {
        for c in 0..size {
            if grid[r][c] != 0 {
                domaines[r][c] = BTreeSet::from([grid[r][c]]);
            }
        }
    }

    fn peers(r: usize, c: usize, size: usize) -> Vec<(usize, usize)> {
        let mut result = Vec::new();
        for k in 0..size {
            if k != c { result.push((r, k)); }
            if k != r { result.push((k, c)); }
        }
        let br = (r / 3) * 3;
        let bc = (c / 3) * 3;
        for i in br..br + 3 {
            for j in bc..bc + 3 {
                if (i, j) != (r, c) {
                    result.push((i, j));
                }
            }
        }
        result
    }

    fn propagate(domaines: &mut Vec<Vec<BTreeSet<i64>>>, size: usize) -> bool {
        let mut changed = true;
        while changed {
            changed = false;
            for r in 0..size {
                for c in 0..size {
                    if domaines[r][c].len() == 1 {
                        let v = *domaines[r][c].iter().next().unwrap();
                        for (pr, pc) in peers(r, c, size) {
                            if domaines[pr][pc].remove(&v) {
                                changed = true;
                                if domaines[pr][pc].is_empty() {
                                    return false;
                                }
                            }
                        }
                    }
                }
            }
        }
        true
    }

    fn select_mrv(domaines: &Vec<Vec<BTreeSet<i64>>>, size: usize) -> Option<(usize, usize)> {
        let mut best: Option<(usize, usize)> = None;
        let mut best_size = size + 1;
        for r in 0..size {
            for c in 0..size {
                let d = &domaines[r][c];
                if 1 < d.len() && d.len() < best_size {
                    best_size = d.len();
                    best = Some((r, c));
                }
            }
        }
        best
    }

    fn backtrack(domaines: &mut Vec<Vec<BTreeSet<i64>>>, size: usize) -> bool {
        if !propagate(domaines, size) {
            return false;
        }
        let cell = select_mrv(domaines, size);
        match cell {
            None => true, // Toutes fixées -> résolu.
            Some((r, c)) => {
                let snapshot: Vec<Vec<BTreeSet<i64>>> =
                    domaines.iter().map(|row| row.clone()).collect();
                let candidates: Vec<i64> = domaines[r][c].iter().cloned().collect();
                for v in candidates {
                    // Restaurer le snapshot puis fixer (r,c) à v.
                    for i in 0..size {
                        for j in 0..size {
                            domaines[i][j] = snapshot[i][j].clone();
                        }
                    }
                    domaines[r][c] = BTreeSet::from([v]);
                    if backtrack(domaines, size) {
                        return true;
                    }
                }
                false
            }
        }
    }

    let solved = backtrack(&mut domaines, size);
    if solved {
        for r in 0..size {
            for c in 0..size {
                grid[r][c] = *domaines[r][c].iter().next().unwrap();
            }
        }
    }
    solved
}

println!("solve_sudoku_mdd défini (propagation de domaines + backtracking MRV).");

solve_sudoku_mdd défini (propagation de domaines + backtracking MRV).


### 6.3 Test du solveur

Résolvons une grille de Sudoku classique.

In [14]:
use std::collections::BTreeSet;
use std::time::Instant;

fn solve_sudoku_mdd(grid: &mut [Vec<i64>], size: usize) -> bool {
    let mut domaines: Vec<Vec<BTreeSet<i64>>> = (0..size)
        .map(|_| (0..size).map(|_| (1..=size as i64).collect()).collect())
        .collect();
    for r in 0..size {
        for c in 0..size {
            if grid[r][c] != 0 {
                domaines[r][c] = BTreeSet::from([grid[r][c]]);
            }
        }
    }

    fn peers(r: usize, c: usize, size: usize) -> Vec<(usize, usize)> {
        let mut result = Vec::new();
        for k in 0..size {
            if k != c { result.push((r, k)); }
            if k != r { result.push((k, c)); }
        }
        let br = (r / 3) * 3;
        let bc = (c / 3) * 3;
        for i in br..br + 3 {
            for j in bc..bc + 3 {
                if (i, j) != (r, c) {
                    result.push((i, j));
                }
            }
        }
        result
    }

    fn propagate(domaines: &mut Vec<Vec<BTreeSet<i64>>>, size: usize) -> bool {
        let mut changed = true;
        while changed {
            changed = false;
            for r in 0..size {
                for c in 0..size {
                    if domaines[r][c].len() == 1 {
                        let v = *domaines[r][c].iter().next().unwrap();
                        for (pr, pc) in peers(r, c, size) {
                            if domaines[pr][pc].remove(&v) {
                                changed = true;
                                if domaines[pr][pc].is_empty() {
                                    return false;
                                }
                            }
                        }
                    }
                }
            }
        }
        true
    }

    fn select_mrv(domaines: &Vec<Vec<BTreeSet<i64>>>, size: usize) -> Option<(usize, usize)> {
        let mut best: Option<(usize, usize)> = None;
        let mut best_size = size + 1;
        for r in 0..size {
            for c in 0..size {
                let d = &domaines[r][c];
                if 1 < d.len() && d.len() < best_size {
                    best_size = d.len();
                    best = Some((r, c));
                }
            }
        }
        best
    }

    fn backtrack(domaines: &mut Vec<Vec<BTreeSet<i64>>>, size: usize) -> bool {
        if !propagate(domaines, size) {
            return false;
        }
        let cell = select_mrv(domaines, size);
        match cell {
            None => true,
            Some((r, c)) => {
                let snapshot: Vec<Vec<BTreeSet<i64>>> =
                    domaines.iter().map(|row| row.clone()).collect();
                let candidates: Vec<i64> = domaines[r][c].iter().cloned().collect();
                for v in candidates {
                    for i in 0..size {
                        for j in 0..size {
                            domaines[i][j] = snapshot[i][j].clone();
                        }
                    }
                    domaines[r][c] = BTreeSet::from([v]);
                    if backtrack(domaines, size) {
                        return true;
                    }
                }
                false
            }
        }
    }

    let solved = backtrack(&mut domaines, size);
    if solved {
        for r in 0..size {
            for c in 0..size {
                grid[r][c] = *domaines[r][c].iter().next().unwrap();
            }
        }
    }
    solved
}

fn py_bool(b: bool) -> &'static str { if b { "True" } else { "False" } }

// Grille de Sudoku classique (0 = case vide).
let mut puzzle: Vec<Vec<i64>> = vec![
    vec![5, 3, 0, 0, 7, 0, 0, 0, 0],
    vec![6, 0, 0, 1, 9, 5, 0, 0, 0],
    vec![0, 9, 8, 0, 0, 0, 0, 6, 0],
    vec![8, 0, 0, 0, 6, 0, 0, 0, 3],
    vec![4, 0, 0, 8, 0, 3, 0, 0, 1],
    vec![7, 0, 0, 0, 2, 0, 0, 0, 6],
    vec![0, 6, 0, 0, 0, 0, 2, 8, 0],
    vec![0, 0, 0, 4, 1, 9, 0, 0, 5],
    vec![0, 0, 0, 0, 8, 0, 0, 7, 9],
];

let t0 = Instant::now();
let solved = solve_sudoku_mdd(&mut puzzle, 9);
let elapsed = t0.elapsed();
println!("Résolu en {:.1} ms : {}", elapsed.as_secs_f64() * 1000.0, py_bool(solved));
println!("Grille solution :");
for row in &puzzle {
    println!("{}", row.iter().map(|v| v.to_string()).collect::<Vec<_>>().join(" "));
};


Résolu en 0.3 ms : True


Grille solution :


5 3 4 6 7 8 9 1 2


6 7 2 1 9 5 3 4 8


1 9 8 3 4 2 5 6 7


8 5 9 7 6 1 4 2 3


4 2 6 8 5 3 7 9 1


7 1 3 9 2 4 8 5 6


9 6 1 5 3 7 2 8 4


2 8 7 4 1 9 6 3 5


3 4 5 2 8 6 1 7 9


#### Interprétation : solveur MDD

Le solveur combine deux idées :

1. **Propagation de contraintes** (style AC-3) : chaque cellule fixée élimine sa
   valeur chez ses *peers* (même ligne, colonne, bloc). C'est l'esprit du MDD
   comme *filtre de domaines*.
2. **Backtracking avec heuristique MRV** (Minimum Remaining Values) : on remplit
   d'abord la cellule la plus contrainte, ce qui réduit drastiquement le facteur
   de branchement.

> **Note pédagogique** : la structure MDD complète (nœuds `MddNode`) est utilisée
> ci-dessus (§4-5) pour *représenter* les contraintes ; le solveur pragmatique
> (§6) en extrait la sémantique (valeurs distinctes) pour une propagation efficace.
> Une implémentation complète propagerait directement sur la structure MDD
> (MDD-consistency), hors de portée de ce notebook d'introduction.

## Exercice 3 : Comparer BDD et MDD sur des contraintes voisines

Le BDD et le MDD ne disent pas la même chose de la même façon. Comparez leurs
tailles (`count_nodes()`) sur deux contraintes cousines, chacune dans son
encodage naturel — les deux outils sont déjà dans le notebook :

- **BDD** (§2-3) : « exactement une variable vraie parmi N » sur N booléens —
  la fonction de l'Exercice 1, via `exactly_one_of_n` ;
- **MDD** (§4) : « N valeurs distinctes sur N cellules », une variable
  multi-valuée par cellule — via `RowMddBuilder`.

Mesurez les deux, puis commentez le point que la comparaison illustre : le BDD
est **booléen** — dès qu'une cellule Sudoku a 9 valeurs possibles, il faut
9 variables one-hot par cellule et la contrainte composée explose ; le MDD garde
le domaine natif. La comparaison honnête n'est pas « la même contrainte » des
deux côtés (l'encodage booléen complet de « tous différents » sur une ligne est
précisément le blowup à démontrer) : c'est le **prix de l'encodage** pour
exprimer la même information.

In [15]:
// EXERCICE : Comparer les performances BDD vs MDD
fn compare_bdd_mdd() -> Option<()> {
    // TODO etudiant : construisez un BDD pour "exactement une variable vraie
    // parmi N" (N booléens) et un MDD pour "une cellule Sudoku ∈ {1..9} unique".
    // Etape 1 : construisez le BDD (N variables booléennes, via exactly_one_of_n).
    // Etape 2 : construisez le MDD (1 variable à 9 valeurs, via RowMddBuilder).
    // Etape 3 : comparez count_nodes() et commentez la compacité.
    None // TODO etudiant
}

println!("Exercice à compléter (comparaison BDD vs MDD).");

Exercice à compléter (comparaison BDD vs MDD).


## 7. Comparaison Sudoku-12 vs Sudoku-14

### 7.1 Tableau comparatif

| Critère | Sudoku-12 (Z3) | Sudoku-14 (BDD/MDD) |
|---------|----------------|---------------------|
| **Approche** | Solveur SMT générique | Structure de données explicite |
| **Outil** | Z3 (pyz3) | Hand-rolled (ce notebook) |
| **Déclaratif ?** | Oui (contraintes déclarées) | Non (construction explicite) |
| **Pédagogique** | Boîte noire | Internes visibles |
| **Performance** | Excellente (SAT solver CDCL) | Modeste (pas de réduction canonique) |
| **Réutilisation** | Toute contrainte exprimable | Spécialisé booléen/multi-valué |

### 7.2 Quand utiliser quoi ?

- **Z3 (Sudoku-12)** : quand la contrainte est complexe, arithmétique, ou que la
  performance est critique. C'est l'outil SOTA pour la satisfiabilité.
- **BDD/MDD (Sudoku-14)** : quand on veut **comprendre** comment une fonction
  booléenne se factorise, ou qu'on a besoin de **représenter canoniquement** un
  ensemble de solutions (énumération, comptage).

Les deux sont **complémentaires** : Z3 est l'outil industriel, le BDD est l'outil
pédagogique qui éclaire les internes de la résolution de contraintes.

## Synthèse

Ce notebook a présenté une implémentation **hand-rolled** des BDD et MDD en Rust,
jumeau pédagogique du notebook C#. Points clés retenus :

1. **BDD** = graphe acyclique représentant une fonction booléenne, avec réduction
   (deux enfants identiques → on remonte).
2. **Apply** = opération universelle (AND/OR/NOT via une seule fonction récursive
   mémoïsée sur la variable de tête).
3. **MDD** = généralisation multi-valuée, adaptée au Sudoku (cellules ∈ {1..9}).
4. Le solveur pragmatique combine propagation de domaines + backtracking MRV.

> **Parité** : ce notebook Rust est le jumeau de
> [Sudoku-14-BDD-CSharp.ipynb](Sudoku-14-BDD-CSharp.ipynb). Les deux hand-rollent
> les mêmes structures (pas de lib BDD), pour enseigner les internes.
> Cf marathon parité (#4956).

## Exercices supplémentaires

### Exercice 4 : Inspection d'un BDD

Écrivez une fonction `count_paths(bdd)` qui compte le nombre de chemins menant à
la feuille `True` (i.e. le nombre d'assignations satisfaisant la fonction).

### Exercice 5 : Évaluation par lot

Étendez `evaluate` pour prendre une *liste* d'assignations et renvoyer la liste
des résultats. Comparez le coût à une évaluation table-de-vérité naïve.

## Pour aller plus loin

- **Suite de la série** : [Sudoku-15 — Résolution probabiliste (NumPyro)](Sudoku-15-Infer-Python.ipynb).
- **Jumeau C#** : [Sudoku-14-BDD-CSharp.ipynb](Sudoku-14-BDD-CSharp.ipynb) — mêmes structures hand-rolled.
- Bryant, R. E. (1986). *Graph-Based Algorithms for Boolean Function Manipulation*.
- [dd](https://pypi.org/project/dd/) — librairie BDD Python.
- Marathon parité : #4956.

## À retenir

1. Un **BDD** représente une fonction booléenne par un graphe acyclique réduit : deux enfants identiques, on remonte — la table de correspondance rend la construction déterministe.
2. `apply` est l'opération **universelle** : AND, OR et NOT tombent d'une seule fonction récursive mémoïsée sur la variable de tête.
3. Le **MDD** généralise le BDD aux variables multi-valuées — c'est ce qui l'adapte au Sudoku (cellules ∈ {1..9}) ; le MDD de la contrainte « une ligne » comptait **31 nœuds**.
4. Le produit de deux MDD **factorise** : 7 × 7 donne **19 nœuds**, pas 49.
5. Le solveur MDD reste **pragmatique** (propagation de domaines + backtracking MRV) ; Z3 (Sudoku-12) et BDD/MDD (Sudoku-14) sont **complémentaires** — l'un industriel, l'autre pédagogique.

## Vérifiez votre compréhension

1. Pourquoi la règle de réduction « deux enfants identiques → on remonte » garantit-elle qu'un BDD est à la fois compact et canonique ?
2. Le produit de deux MDD de 7 nœuds en donne un de 19 nœuds et non de 49 : que mesure cet écart sur la factorisation des contraintes ?
3. Dans quel cas choisir la représentation BDD/MDD (comptage, représentation canonique) plutôt que le solveur SMT Z3 ?

> Carnet issu de l'EPIC #26534.